In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import pandas as pd

# Load movies fully (small file)
print("Loading movies...")
movies = pd.read_csv('/content/drive/MyDrive/movie_recommendation/movie.csv')
print("Movies shape:", movies.shape)

# Load only 5 million ratings instead of 20 million
print("\nLoading 5 million ratings...")
ratings = pd.read_csv(
    '/content/drive/MyDrive/movie_recommendation/rating.csv',
    nrows=5000000  # only load 5M rows
)
print("Ratings shape:", ratings.shape)

# Merge
print("\nMerging...")
movies_with_ratings = ratings.merge(movies, on='movieId')
del ratings

# Mean and count
ratings_mean_count = movies_with_ratings.groupby('title')['rating'].agg(['mean', 'count'])

# Filter movies with more than 500 ratings
ratings_filtered = ratings_mean_count[ratings_mean_count['count'] > 500]
print("Movies after filtering:", ratings_filtered.shape[0])

# Create matrix
print("\nCreating matrix... please wait...")
user_movie_matrix = movies_with_ratings.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)
del movies_with_ratings

user_movie_matrix_filtered = user_movie_matrix[ratings_filtered.index]
del user_movie_matrix

user_movie_matrix_filled = user_movie_matrix_filtered.fillna(0)
del user_movie_matrix_filtered

print("Matrix shape:", user_movie_matrix_filled.shape)
print("\nAll done!")

Loading movies...
Movies shape: (27278, 3)

Loading 5 million ratings...
Ratings shape: (5000000, 4)

Merging...
Movies after filtering: 2035

Creating matrix... please wait...
Matrix shape: (34395, 2035)

All done!


In [2]:
def recommend(movie_name):
    movie_name = movie_name.strip()

    if movie_name not in user_movie_matrix_filled.columns:
        return "Movie not found, please check the name."

    movie_ratings = user_movie_matrix_filled[movie_name]

    similar = user_movie_matrix_filled.corrwith(movie_ratings)

    corr_df = pd.DataFrame(similar, columns=['correlation'])
    corr_df.dropna(inplace=True)

    corr_df = corr_df.join(ratings_mean_count['count'])

    recommendations = corr_df[corr_df['count'] > 500].sort_values('correlation', ascending=False)

    return recommendations.iloc[1:11]

# Test it
print(recommend('Toy Story (1995)'))

                                                   correlation  count
title                                                                
Toy Story 2 (1999)                                    0.415284   5658
Willy Wonka & the Chocolate Factory (1971)            0.363468   7066
Independence Day (a.k.a. ID4) (1996)                  0.360387  11769
Mission: Impossible (1996)                            0.338090   9297
Star Wars: Episode IV - A New Hope (1977)             0.332978  13672
Bug's Life, A (1998)                                  0.332018   5053
Back to the Future (1985)                             0.328474  10416
Star Wars: Episode VI - Return of the Jedi (1983)     0.325124  11781
Lion King, The (1994)                                 0.322638   9687
Aladdin (1992)                                        0.322363  10334


In [3]:
def find_movie_name(user_input):
    user_input = user_input.lower()
    matches = []
    for title in user_movie_matrix_filled.columns:
        if user_input in title.lower():
            matches.append(title)
    return matches

def get_recommendations(user_input):
    matches = find_movie_name(user_input)

    if len(matches) == 0:
        print("No matching movie found.")

    elif len(matches) == 1:
        print("Using:", matches[0])
        print("\nRecommendations:")
        print(recommend(matches[0]))

    else:
        print("Did you mean:\n")
        for i, movie in enumerate(matches[:10]):
            print(f"{i+1}. {movie}")

        choice = int(input("\nEnter number: "))
        selected_movie = matches[choice - 1]
        print("\nRecommendations for:", selected_movie)
        print(recommend(selected_movie))

# Test it
get_recommendations('matrix')

Did you mean:

1. Animatrix, The (2003)
2. Matrix Reloaded, The (2003)
3. Matrix Revolutions, The (2003)
4. Matrix, The (1999)

Enter number: 4

Recommendations for: Matrix, The (1999)
                                                    correlation  count
title                                                                 
Lord of the Rings: The Fellowship of the Ring, ...     0.518659   9369
Fight Club (1999)                                      0.513453   9947
Sixth Sense, The (1999)                                0.497587   9697
Star Wars: Episode V - The Empire Strikes Back ...     0.484725  11361
Men in Black (a.k.a. MIB) (1997)                       0.483776   9015
Lord of the Rings: The Two Towers, The (2002)          0.481924   8474
Lord of the Rings: The Return of the King, The ...     0.481458   7871
Gladiator (2000)                                       0.476713   8273
Fifth Element, The (1997)                              0.473929   6984
Terminator, The (1984)            

In [6]:
import pickle

# Save the matrix
print("Saving user_movie_matrix_filled...")
with open('/content/drive/MyDrive/movie_recommendation/user_movie_matrix_filled.pkl', 'wb') as f:
    pickle.dump(user_movie_matrix_filled, f)

# Save ratings_mean_count
print("Saving ratings_mean_count...")
with open('/content/drive/MyDrive/movie_recommendation/ratings_mean_count.pkl', 'wb') as f:
    pickle.dump(ratings_mean_count, f)

# Save movies list for streamlit dropdown
print("Saving movies list...")
movies_list = list(user_movie_matrix_filled.columns)
with open('/content/drive/MyDrive/movie_recommendation/movies_list.pkl', 'wb') as f:
    pickle.dump(movies_list, f)

print("\nAll pickle files saved successfully!")
print("Files saved:")
print(os.listdir('/content/drive/MyDrive/movie_recommendation'))

Saving user_movie_matrix_filled...
Saving ratings_mean_count...
Saving movies list...

All pickle files saved successfully!
Files saved:
['rating.csv', 'movie.csv', 'user_movie_matrix_filled.pkl', 'ratings_mean_count.pkl', 'movies_list.pkl']


In [5]:
import os

print("Files saved:")
print(os.listdir('/content/drive/MyDrive/movie_recommendation'))

Files saved:
['rating.csv', 'movie.csv', 'user_movie_matrix_filled.pkl', 'ratings_mean_count.pkl', 'movies_list.pkl']


In [7]:
from google.colab import files

print("Downloading user_movie_matrix_filled.pkl...")
files.download('/content/drive/MyDrive/movie_recommendation/user_movie_matrix_filled.pkl')

print("Downloading ratings_mean_count.pkl...")
files.download('/content/drive/MyDrive/movie_recommendation/ratings_mean_count.pkl')

print("Downloading movies_list.pkl...")
files.download('/content/drive/MyDrive/movie_recommendation/movies_list.pkl')

print("All files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files downloaded!
